**`AI SOCIAL MEDIA CONTENT GENERATOR`**

With Gradio UI · Emoji & Hashtag Options · Engagement Score Strategy

In [86]:
%pip install -q langchain langchain-groq langchain-core gradio


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [87]:
import os
import gradio as gr
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [88]:
linkedin_prompt = PromptTemplate(
    input_variables=["topic", "word_count", "use_emojis", "use_hashtags", "hashtag_count"],
    template="""Hey AI! I need your help writing a professional LinkedIn post about: {topic}

Keep it professional yet engaging – like a conversation between industry experts.
1. Start with a hook that makes people stop scrolling.
2. Share 2-3 valuable insights.
3. End with a clear Call-To-Action (CTA) to spark a discussion.
4. Target LENGTH: STICK CLOSELY TO {word_count} WORDS. 
   - This is a WORD count (number of words), NOT a character count. 
   - If {word_count} is 100, write a long paragraph (around 6-8 sentences). 
   - Expand your insights thoroughly to reach this word count.
5. Emojis: {use_emojis}
6. Hashtags: {use_hashtags} - if yes, please add exactly {hashtag_count} relevant ones at the very end.

Also, please calculate an 'Engagement Score' (0-100) based on how catchy the hook is and how likely people are to comment and give me a brief explanation of that score at the bottom!

"""
)

instagram_prompt = PromptTemplate(
    input_variables=["topic", "word_count", "use_emojis", "use_hashtags", "hashtag_count"],
    template="""Write a cool Instagram caption about: {topic}

We want high energy and a conversational vibe!
1. Start with something relatable or a bold statement.
2. Use a friendly and informal tone.
3. Ask a fun question at the end to get people talking in the comments.
4. Target LENGTH: STICK CLOSELY TO {word_count} WORDS. 
   - This is a WORD count (number of words), NOT a character count. 
   - If {word_count} is 100, write a clear, detailed story or description. 
   - Do not stop until you have reached approximately {word_count} individual words.
5. Emojis: {use_emojis}
6. Hashtags: {use_hashtags} - if yes, include exactly {hashtag_count} hashtags.

Don't forget to include an 'Engagement Strategy' score at the bottom based on how likely this is to get likes and comments, along with a quick tip to boost engagement!

"""
)

x_prompt = PromptTemplate(
    input_variables=["topic", "word_count", "use_emojis", "use_hashtags", "hashtag_count"],
    template="""Draft a punchy X (Twitter) post about: {topic}

Think viral vibes – short, shareable, and bold!
1. Start with a punchy hook.
2. Use short sentences for better readability.
3. Stay within the limits while being impactful.
4. Target LENGTH: STICK CLOSELY TO {word_count} WORDS. 
   - This is a WORD count (number of words), NOT a character count. 
   - If {word_count} is high (e.g., 200), write a detailed thread or a long-form post. 
   - Ensure the total word count is approximately {word_count} words.
5. Emojis: {use_emojis}
6. Hashtags: {use_hashtags} - include exactly {hashtag_count} hashtags.

Show me the Engagement Score based on how likely this is to go viral and a quick tip to increase shareability!

"""
)

facebook_prompt = PromptTemplate(
    input_variables=["topic", "word_count", "use_emojis", "use_hashtags", "hashtag_count"],
    template="""Create a warm and friendly Facebook post about: {topic}

The goal is to connect with a community!
1. Use a warm, conversational tone.
2. Encourage readers to share their own thoughts or stories.
3. End with an open-ended question that's easy to answer.
4. Target LENGTH: STICK CLOSELY TO {word_count} WORDS. 
   - This is a WORD count (number of words), NOT a character count. 
   - Provide enough storytelling and detail to meet the {word_count} word budget.
5. Emojis: {use_emojis}
6. Hashtags: {use_hashtags} - if yes, add exactly {hashtag_count} hashtags.

Include an estimated Engagement Score and a small tip for community building and fostering discussion!
"""
)

threads_prompt = PromptTemplate(
    input_variables=["topic", "word_count", "use_emojis", "use_hashtags", "hashtag_count"],
    template="""Write an authentic Threads post about: {topic}

Keep it casual, like you're talking to a friend!
1. Share a genuine opinion or a quick story.
2. Spark a debate or a thoughtful discussion.
3. Avoid sounding like an advertisement – stay real.
4. Target LENGTH: STICK CLOSELY TO {word_count} WORDS. 
   - This is a WORD count (number of words), NOT a character count. 
   - Elaborate on your points to ensure you reach the {word_count} word mark.
5. Emojis: {use_emojis}
6. Hashtags: {use_hashtags} - if yes, add exactly {hashtag_count} hashtags.

Provide an Engagement Score based on the 'discussion potential' of the post and a quick tip to encourage more replies and interactions!

"""
)

youtube_prompt = PromptTemplate(
    input_variables=["topic", "word_count", "use_emojis", "use_hashtags", "hashtag_count"],
    template="""Write a YouTube Community post about: {topic}

Address our subscribers directly!
1. Build excitement for a new video or a topic.
2. Use a sense of 'behind the scenes' or 'inside scoop'.
3. Encourage likes and interactions to boost the algorithm.
4. Target LENGTH: STICK CLOSELY TO {word_count} WORDS. 
   - This is a WORD count (number of words), NOT a character count. 
   - Expand on the details significantly to meet the {word_count} word requirement.
5. Emojis: {use_emojis}
6. Hashtags: {use_hashtags} - if yes, include exactly {hashtag_count} hashtags.

Give me an Engagement Score and a tip for YouTube growth based on how likely this is to get likes and comments!
"""
)

platform_options = {
    "linkedin":  linkedin_prompt,
    "instagram": instagram_prompt,
    "x":         x_prompt,
    "facebook":  facebook_prompt,
    "threads":   threads_prompt,
    "youtube":   youtube_prompt,
}


In [89]:
parser = StrOutputParser()

def create_social_post(api_key, topic, platform, word_limit, add_emojis, add_hashtags, hashtag_count):
    if not api_key:
        return "Hey! You forgot to enter your Groq API Key above. "
    if not topic.strip():
        return "I need a topic to write about! Please enter something in the box. "

    try:
        ai_brain = ChatGroq(
            model="llama-3.3-70b-versatile", 
            temperature=0.7,
            api_key=api_key
        )
        
        prompt_to_use = platform_options[platform.lower()]
        
        content_pipeline = prompt_to_use | ai_brain | parser
        
        final_instructions = {
            "topic":         topic,
            "word_count":    word_limit,
            "use_emojis":    "Feel free to use plenty of emojis! " if add_emojis else "Strictly no emojis.",
            "use_hashtags":  "Yes, please add some hashtags." if add_hashtags else "No hashtags needed.",
            "hashtag_count": hashtag_count
        }
        
        return content_pipeline.invoke(final_instructions)

    except Exception as error:
        return f"Something went wrong: {str(error)}"


In [90]:
with gr.Blocks() as my_app:
    gr.Markdown("# Social Media Content Generator")
    
    api_input = gr.Textbox(label="API Key", type="password")
    topic_input = gr.Textbox(label="Topic", lines=2)
    
    with gr.Row():
        platform_choice = gr.Dropdown(
            choices=["LinkedIn", "Instagram", "X", "Facebook", "Threads", "YouTube"],
            value="LinkedIn",
            label="Platform"
        )
        length_input = gr.Textbox(label="Word Count", value="100")

    with gr.Row():
        emoji_toggle = gr.Checkbox(label="Include Emojis", value=True)
        hashtag_toggle = gr.Checkbox(label="Include Hashtags", value=True)
        count_input = gr.Textbox(label="Hashtag Count", value="3")

    submit_btn = gr.Button("Generate")

    output_text = gr.Textbox(label="Result", lines=15)

    hashtag_toggle.change(
        fn=lambda x: gr.update(visible=x),
        inputs=hashtag_toggle,
        outputs=count_input
    )

    submit_btn.click(
        fn=create_social_post,
        inputs=[api_input, topic_input, platform_choice, length_input, emoji_toggle, hashtag_toggle, count_input],
        outputs=[output_text]
    )

custom_styling = """
* { 
    transition: none !important; 
    animation: none !important; 
    transform: none !important;
}
footer {visibility: hidden}
"""

my_app.launch(css=custom_styling)


* Running on local URL:  http://127.0.0.1:7897
* To create a public link, set `share=True` in `launch()`.
